In [1]:
import json
import numpy as np
from tqdm.auto import tqdm


def recall_at_k(results, gt_path, k):
    retrieved = [r["path"] for r in results[:k]]
    return int(gt_path in retrieved)


def reciprocal_rank(results, gt_path):
    for rank, result in enumerate(results, start=1):
        if result["path"] == gt_path:
            return 1.0 / rank
    return 0.0


def ndcg_at_k(results, gt_path, k):
    for rank, result in enumerate(results[:k], start=1):
        if result["path"] == gt_path:
            return 1.0 / np.log2(rank + 1)
    return 0.0


def evaluate(annotation_file, retrieve_fn):
    with open(annotation_file, "r") as f:
        dataset = [
            json.loads(line)
            for line in f
            if line.strip()
        ]
    metrics = {
        "Recall@1": [],
        "Recall@5": [],
        "Recall@10": [],
        "NDCG@5": [],
        "NDCG@10": [],
        "MRR": [],
    }

    for sample in tqdm(dataset, total=len(dataset), desc="Evaluating"):
        gt = sample["image_path"]
        for caption in sample["captions"]:
            results = retrieve_fn(caption)
            metrics["Recall@1"].append(
                recall_at_k(results, gt, 1)
            )
            metrics["Recall@5"].append(
                recall_at_k(results, gt, 5)
            )
            metrics["Recall@10"].append(
                recall_at_k(results, gt, 10)
            )
            metrics["NDCG@5"].append(
                ndcg_at_k(results, gt, 5)
            )
            metrics["NDCG@10"].append(
                ndcg_at_k(results, gt, 10)
            )
            metrics["MRR"].append(
                reciprocal_rank(results, gt)
            )

    return {
        k: np.mean(v)
        for k, v in metrics.items()
    }

/Users/armaanjagirdar/glance_task/env2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
metrics = evaluate(
    "dataset/fashionpedia_captions.jsonl",
    lambda q: search_engine.query(
        q,
        k_initial=50,
        k_final=20,   
    ),
)

print(metrics)

Evaluating:   1%|          | 6/1158 [01:27<4:39:06, 14.54s/it]


KeyboardInterrupt: 

In [ ]:
import time

t0 = time.perf_counter()
initial_results = self._retrieve_initial(query_text, k=k_initial)
t1 = time.perf_counter()

reranked_results = []

with torch.no_grad():
    for start in range(0, len(initial_results), batch_size):

        batch = initial_results[start:start + batch_size]

        images = [
            self.image_cache[r["path"]]
            for r in batch
        ]

        t2 = time.perf_counter()

        inputs = self.embedder.cross_processor(
            images=images,
            text=[query_text] * len(images),
            return_tensors="pt",
            padding=True,
        ).to(self.embedder.device)

        t3 = time.perf_counter()

        outputs = self.embedder.cross_model(**inputs)

        t4 = time.perf_counter()

print(f"HNSW      : {t1-t0:.3f}s")
print(f"Processor : {t3-t2:.3f}s")
print(f"BLIP      : {t4-t3:.3f}s")